# Mettler Toledo WXS205SDU

The WXS205SDU is a high-precision automated weigh module from Mettler Toledo, commonly used for gravimetric liquid transfer verification (e.g. in the Hamilton Liquid Verification Kit).

| Property | Value |
|---|---|
| [OEM Link](https://www.mt.com/gb/en/home/products/Industrial_Weighing_Solutions/high-precision-weigh-sensors/weigh-module-wxs205sdu-15-11121008.html) | |
| Communication | Serial / RS-232 |
| VID:PID | `0x0403:0x6001` |
| Load range | 0 -- 220 g |
| Readability | 0.1 mg |

The backend has been tested on the WXS205SDU but, per Mettler Toledo firmware documentation, should be applicable to other "Automated Precision Weigh Modules" in the WX and WMS series.

**Capabilities:** [Weighing](../../capabilities/weighing)

## Physical setup

The system consists of two required units and one optional unit:

| Component | Required | Description |
|---|---|---|
| Load Cell | yes | The weighing platform where samples are placed |
| Electronic Unit | yes | The control and communication module |
| Terminal/Display | no | For manual reading; not needed when using PyLabRobot |

Connect the electronic unit to your computer via the RS-232 serial port. You will likely need a USB-to-serial adapter (any generic FTDI-based adapter should work).

```{warning}
The scale requires a warm-up period after being powered on. Mettler Toledo specifies 60--90 minutes, though 30 minutes is often sufficient in practice. If you attempt measurements before warm-up, you may see: *"Command understood but currently not executable"*.
```

## Setup

In [ ]:
from pylabrobot.mettler_toledo import MettlerToledoWXS205SDUDriver, MettlerToledoWXS205SDUScaleBackend
from pylabrobot.capabilities.weighing import Scale

driver = MettlerToledoWXS205SDUDriver(port="/dev/cu.usbserial-110")  # replace with your port
backend = MettlerToledoWXS205SDUScaleBackend(driver=driver)
scale = Scale(backend=backend)

await driver.setup()
await scale.setup()

## Weighing

The scale exposes the standard [Weighing](../../capabilities/weighing) capability: `zero()`, `tare()`, and `read_weight()`.

### Zero

Calibrates the scale to read zero with an empty platform. Use at the start of a workflow.

In [ ]:
await scale.zero()

### Tare

Resets the displayed weight to zero while accounting for a container already on the platform. Place your container, then tare, so subsequent readings reflect only the added material.

In [ ]:
await scale.tare()

### Read weight

Returns the current weight in grams.

In [ ]:
weight = await scale.read_weight()
print(f"Weight: {weight} g")

### Backend-specific methods

The backend exposes additional methods beyond the standard capability interface. You can access them through `scale.backend`:

- `request_tare_weight()` -- retrieve the stored tare value
- `request_serial_number()` -- read the scale's serial number
- `clear_tare()` -- clear the stored tare weight
- `zero(timeout=...)` / `tare(timeout=...)` / `read_weight(timeout=...)` -- pass a timeout mode (`"stable"`, `0`, or seconds). See the [Weighing capability docs](../../capabilities/weighing) for details on timeout modes.

In [ ]:
tare_value = await scale.backend.request_tare_weight()
print(f"Stored tare: {tare_value} g")

## Teardown

In [ ]:
await scale.stop()
await driver.stop()